# Velocity Kinematics and Statics

Source orientation: printed pages 171-218, PDF pages 189-236. This notebook is an original visualization-first lesson inspired by the structure of *Modern Robotics: Mechanics, Planning, and Control* by Kevin M. Lynch and Frank C. Park. It does not quote or reproduce textbook prose, exercises, screenshots, or page crops.

## Chapter Question

What does the Jacobian say about instantaneous motion, lost directions, and static force balance?

This question is the thread for the chapter. The goal is not to memorize a list of formulas; it is to make the geometry inspectable. A robot mechanism is a physical machine, but the way we reason about it is through spaces, maps, constraints, tangents, and dual forces. The notebook keeps those objects visible. Every diagram and computation below is designed to answer a local question that a reader can check: what is moving, what is fixed, what map is being applied, what invariant should survive, and where can the model fail?

## Translation Guide

The source chapter or appendix is translated into computational language using these terms: space Jacobian, body Jacobian, singularity, manipulability, wrench, virtual work, rank. The important conversions for this notebook are:

- A Jacobian maps joint rates to a twist.
- The transpose maps endpoint wrench to joint torques by virtual work.
- Singularities are rank events, not just bad numbers.

The central habit is to name the representation and the invariant at the same time. Coordinates are useful only when we know what geometric object they represent. A column of joint angles may describe a point on a torus, a homogeneous transform may describe a rigid frame, and a matrix rank may reveal an instantaneous loss of motion. The notebook therefore pairs each formula with a small experiment: a plot, a residual, an ellipsoid, a path, a graph, or a constraint surface.

## Route Through the Notebook

1. Establish the objects and conventions needed for velocity kinematics and statics.
2. Build a concept dependency map so definitions have visible structure.
3. Generate the main visual lab: Jacobians, singularities, statics, and manipulability.
4. Run a compact worked example that exposes an invariant.
5. Try an applied lab that changes a parameter and asks what should remain true.
6. Finish with sanity checks that assert artifact existence, image variation, and numerical margins.

This is a standalone course notebook. The PDF can be useful for historical context and exercises, but the lesson here is complete without it. Definitions are restated in fresh language, examples are computed from scratch, and visuals are generated by course-local code. When the notebook mentions a source span, it is a navigation reference rather than a dependency for comprehension.

## Visualization Storyboard

| Concept | Representation | Artifact | Inspection target |
| --- | --- | --- | --- |
| concept dependency map | directed graph | `artifacts/chapter-05-velocity-kinematics-and-statics/figures/concept-dependency-map.png` | which definitions feed the chapter's computation |
| Jacobians, singularities, statics, and manipulability | Jacobian image and velocity ellipsoid | `artifacts/chapter-05-velocity-kinematics-and-statics/figures/velocity-kinematics-lab.png` | where rank and ellipsoid shape reveal available motion and force directions |
| numeric invariant check | residual bar chart and JSON summary | `artifacts/chapter-05-velocity-kinematics-and-statics/figures/velocity-kinematics-checks.png` | small residuals, positive margins, or rank changes |

The first visual is a dependency map. It is intentionally simple: before computing anything, the reader should see which concepts support which later moves. The second visual is the main lab for this chapter. It turns the chapter's core geometry into something that can be inspected rather than imagined. The third visual is a numeric check: a residual, rank, eigenvalue, path length, or comparable margin that can be tested after the figure is built.

## Working Principles

The most reliable way to learn robotics geometry is to move between three views. The first view is symbolic: equations name maps and constraints. The second view is numerical: a small instance exposes scale, rank, conditioning, and residuals. The third view is visual: geometry becomes a shape, path, ellipsoid, cone, graph, or surface. This notebook keeps all three views close together. If a symbolic statement is correct, the numerical experiment should produce the expected small residual or positive margin. If a visual is meaningful, it should make a specific invariant or failure mode easier to see.

For velocity kinematics and statics, the relevant failure modes are not side details; they are part of the concept. Singularities, chart boundaries, rank loss, time-scaling limits, contact mode changes, and wheel constraints are all examples of geometry asserting itself. A robust robotics model does not pretend those edges are absent. It names them, draws them, and then tests a small case so the reader can recognize the issue in later code.

## Applied Lab

Sweep a planar arm near singularity and watch the velocity ellipsoid collapse.

In the lab, vary one parameter at a time and predict the direction of change before running the code. For example, ask whether a rank should change, whether a path should lengthen, whether an eigenvalue should stay positive, whether a cone should widen, or whether a coordinate chart should approach a singular value. This prediction step is small, but it is what turns a figure into a mathematical instrument.

## Pitfalls To Watch

- Large joint speeds near singularity do not imply large endpoint freedom.
- Manipulability is representation dependent unless the metric is stated.

These pitfalls are deliberately operational. If a computation becomes confusing, check frame labels, units, rank, and the distinction between a physical object and its coordinates. Many robotics errors are not arithmetic mistakes; they are mismatches between a representation and the geometry it claims to encode.

## Takeaways

- Instantaneous kinematics is linear geometry around a configuration.
- Statics is the dual linear map seen through power.

By the end of this notebook, the reader should be able to explain the chapter's main object, build a small computed example, interpret the generated visual artifact, and state at least one numerical sanity check that would catch an implementation mistake.

In [ ]:
from pathlib import Path
import json
import math
import sys

import numpy as np

HERE = Path.cwd().resolve()
BOOK_ROOT = next(
    parent for parent in [HERE, *HERE.parents]
    if (parent / "AGENTS.md").exists() and (parent / "Mordern Robotics.pdf").exists()
)
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import display_artifact, save_json
from utils.validation import assert_artifact, image_stats

print(f"BOOK_ROOT={BOOK_ROOT}")

In [ ]:
CHAPTER = {
  "number": 5,
  "slug": "chapter-05-velocity-kinematics-and-statics",
  "title": "Velocity Kinematics and Statics",
  "notebook": "05-velocity-kinematics-and-statics.ipynb",
  "printed_start": 171,
  "printed_end": 218,
  "pdf_start": 189,
  "pdf_end": 236,
  "part_slug": "part-02-manipulator-kinematics",
  "part_title": "Manipulator Kinematics",
  "theme": "kinematics",
  "visual_focus": "Jacobians, singularities, statics, and manipulability",
  "visual_kind": "Jacobian image and velocity ellipsoid",
  "artifact_stem": "velocity-kinematics",
  "inspection_target": "where rank and ellipsoid shape reveal available motion and force directions",
  "question": "What does the Jacobian say about instantaneous motion, lost directions, and static force balance?",
  "terms": [
    "space Jacobian",
    "body Jacobian",
    "singularity",
    "manipulability",
    "wrench",
    "virtual work",
    "rank"
  ],
  "translation": [
    "A Jacobian maps joint rates to a twist.",
    "The transpose maps endpoint wrench to joint torques by virtual work.",
    "Singularities are rank events, not just bad numbers."
  ],
  "lab": "Sweep a planar arm near singularity and watch the velocity ellipsoid collapse.",
  "pitfalls": [
    "Large joint speeds near singularity do not imply large endpoint freedom.",
    "Manipulability is representation dependent unless the metric is stated."
  ],
  "takeaways": [
    "Instantaneous kinematics is linear geometry around a configuration.",
    "Statics is the dual linear map seen through power."
  ]
}

from utils.visuals import build_storyboard
storyboard = build_storyboard(CHAPTER)
storyboard

In [ ]:
from utils.visuals import build_chapter_visuals

outputs = build_chapter_visuals(CHAPTER)
for artifact in outputs['figures']:
    display_artifact(artifact, width=760)
outputs['metrics']

## Worked Example

The worked example below is intentionally compact and reusable. It checks a planar arm Jacobian, a rotation exponential/logarithm round trip, and the twist/wrench power pairing under a frame transform. These are small tests, but they exercise the same representation discipline used throughout the course.

For this chapter, the key habit is to separate the physical statement from the coordinates used to compute it. The planar Jacobian reports the instantaneous endpoint velocity produced by each joint rate at one configuration; its rank tells whether the endpoint can move in both local planar directions. The power check uses the same idea on the dual side: a twist and a wrench may be re-expressed in another frame, but their scalar pairing should not change. If that number changes, the bug is usually a mismatched adjoint, an inverted wrench transform, or a frame label that was silently swapped.

In [ ]:
from utils.kinematics import planar_arm_points, planar_jacobian, manipulability_measure
from utils.lie import adjoint, se3_exp, se3_log, so3_exp, so3_log, transform_from, wrench_power

lengths = np.array([1.0, 0.75, 0.45])
theta = np.array([0.45, -0.8, 0.6])
points = planar_arm_points(lengths, theta)
J = planar_jacobian(lengths, theta)
R = so3_exp(np.array([0.2, -0.1, 0.35]))
round_trip = np.linalg.norm(so3_log(R) - np.array([0.2, -0.1, 0.35]))
T = transform_from(R, np.array([0.25, -0.1, 0.4]))
twist = np.array([0.1, -0.2, 0.3, 0.4, 0.2, -0.1])
wrench = np.array([0.3, 0.1, -0.2, 1.0, -0.4, 0.2])
power_before = wrench_power(twist, wrench)
power_after = wrench_power(adjoint(T) @ twist, np.linalg.inv(adjoint(T)).T @ wrench)
worked_example = {
    "endpoint": points[-1].round(4).tolist(),
    "jacobian_rank": int(np.linalg.matrix_rank(J)),
    "manipulability": float(manipulability_measure(J)),
    "rotation_log_round_trip": float(round_trip),
    "power_pairing_error": float(abs(power_before - power_after)),
}
worked_example

## Applied Lab

The applied lab sweeps the second joint of a two-link arm while the first joint stays fixed. The numerical summary should agree with the visual lab: the maximum manipulability occurs away from a straight or folded arm, while the minimum drops near the singular postures where the velocity ellipse collapses toward a line.

Before running the cell, predict what the endpoints of the sweep should do to the Jacobian columns. When the two links nearly align, both joints mostly push the endpoint in the same instantaneous direction, so the column span loses area. When the links bend, the columns become more independent and the determinant-like manipulability measure grows. This is the local geometry behind the practical warning that inverse velocity commands become fragile near singularity.

In [ ]:
from utils.control import pd_response
from utils.dynamics import two_link_mass_matrix
from utils.grasping import friction_cone, grasp_matrix
from utils.mobile import mecanum_wheel_matrix, unicycle_rollout
from utils.planning import cubic_time_scaling, dijkstra_grid, quintic_time_scaling

theme = CHAPTER["theme"]
if theme in {"configuration", "kinematics"}:
    sweep = np.linspace(-np.pi, np.pi, 90)
    values = [manipulability_measure(planar_jacobian(np.array([1.0, 0.8]), np.array([0.25, q]))) for q in sweep]
    lab_summary = {"theme": theme, "max_manipulability": float(np.max(values)), "min_manipulability": float(np.min(values))}
elif theme in {"rigid", "appendix"}:
    angles = np.linspace(0.0, 0.95 * np.pi, 60)
    residuals = [np.linalg.norm(so3_log(so3_exp(np.array([0.0, 0.0, a]))) - np.array([0.0, 0.0, a])) for a in angles]
    lab_summary = {"theme": theme, "max_rotation_residual": float(np.max(residuals))}
elif theme == "dynamics":
    eigs = np.array([np.linalg.eigvalsh(two_link_mass_matrix(q)) for q in np.linspace(-np.pi, np.pi, 80)])
    lab_summary = {"theme": theme, "minimum_mass_eigenvalue": float(eigs.min())}
elif theme == "planning":
    t = np.linspace(0, 1, 100)
    lab_summary = {"theme": theme, "cubic_midpoint": float(cubic_time_scaling(np.array([0.5]), 1.0)[0]), "quintic_midpoint": float(quintic_time_scaling(np.array([0.5]), 1.0)[0])}
elif theme == "contact":
    cone = friction_cone(np.array([0.0, 1.0]), 0.5)
    G = grasp_matrix(np.array([[1.0, 0.0], [-0.5, 0.86], [-0.5, -0.86]]), np.array([[-1.0, 0.0], [0.5, -0.86], [0.5, 0.86]]))
    lab_summary = {"theme": theme, "cone_samples": int(len(cone)), "grasp_rank": int(np.linalg.matrix_rank(G))}
elif theme == "mobile":
    controls = np.c_[np.ones(80) * 0.5, np.ones(80) * 0.3]
    path = unicycle_rollout(controls)
    lab_summary = {"theme": theme, "wheel_map_rank": int(np.linalg.matrix_rank(mecanum_wheel_matrix())), "final_pose": path[-1].round(4).tolist()}
else:
    lab_summary = {"theme": theme}

lab_summary

## Sanity Checks

The final cell asserts that the generated artifacts exist, are large enough to be useful, and have nontrivial pixel variation. It also stores a JSON summary under the chapter's artifact subtree so the notebook leaves a machine-checkable trace.

In [ ]:
artifact_stats = {}
for artifact in outputs['figures']:
    resolved = assert_artifact(artifact)
    stats = image_stats(resolved)
    assert stats['pixel_std'] > 2.0, f'low variation image: {resolved}'
    artifact_stats[resolved.name] = stats
assert_artifact(outputs['checks'], min_size=100)
sanity = {'chapter': CHAPTER['slug'], 'metrics': outputs['metrics'], 'worked_example': worked_example, 'lab_summary': lab_summary, 'artifact_stats': artifact_stats}
sanity_path = save_json(sanity, CHAPTER['slug'], 'checks', 'notebook-sanity.json')
display_artifact(sanity_path)
sanity